# 01d — Extract OSM Pollution Proximity (Overpass API)
**Data source:** [Overpass API](http://overpass-api.de/) — OpenStreetMap features

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00 output

**Output:** `osm.parquet` (one row per unique station)

### API Resilience & Optimizations
1. **Batching:** We bundle all 12 queries (4 categories x 3 radii) into a single batch query per station, reducing calls from 2,040 to 170.
2. **Global Mirror Failover:** We rotate queries through multiple public **global** mirrors (lz4, z, Kumi Systems) upon retries. Regional-only mirrors are excluded.
3. **User-Agent Spoofing:** Overpass servers block default Python requests headers with `406 Not Acceptable` to prevent scraping. We set custom headers (`User-Agent: curl/8.7.1` and `Accept: */*`) to guarantee successful API responses.
4. **Verbose Logging & Status Tracking:** Log outputs show exact feature counts in real time, and we add an `osm_status` column (`Success` or `Failed`) to identify which queries failed all retries for local-Kaggle merging.

**Estimated time:** ~5-8 min for ~170 stations (with rate-limiting safeguards)

In [ ]:
# Install required packages if running in Kaggle environment
!pip install -q geopandas pyarrow requests tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, time, requests, logging
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01d_osm')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column config
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

In [ ]:
# Load base data to get unique stations
base_train_path = find_file('train_base.parquet')
base_val_path   = find_file('val_base.parquet')

train_base = pd.read_parquet(base_train_path)
val_base   = pd.read_parquet(base_val_path)
all_data   = pd.concat([train_base, val_base], ignore_index=True)

unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
log.info(f'Loaded unique stations: {len(unique_stations)}')

---
## Optimized Overpass Batch Query with Endpoint Rotation

In [ ]:
OVERPASS_ENDPOINTS = [
    'http://overpass-api.de/api/interpreter',          # Main (DE)
    'https://lz4.overpass-api.de/api/interpreter',     # Mirror lz4 (DE)
    'https://z.overpass-api.de/api/interpreter',       # Mirror z (DE)
    'https://overpass.kumi.systems/api/interpreter'    # Mirror Kumi (DE)
]

def fetch_osm_station(lat, lon, station_id, station_num, total_stations, retries=4):
    """
    Fetches all 12 OSM counts (4 categories x 3 radii) in a single batch query.
    Rotates endpoints on failure to bypass server-specific blockages.
    """
    query = f"""[out:json][timeout:90];
    node(around:1000,{lat},{lon})["man_made"="mine"]->.m1;.m1 out count;
    way(around:1000,{lat},{lon})["man_made"="wastewater_plant"]->.w1;.w1 out count;
    way(around:1000,{lat},{lon})["landuse"="farmland"]->.f1;.f1 out count;
    way(around:1000,{lat},{lon})["highway"]->.h1;.h1 out count;

    node(around:5000,{lat},{lon})["man_made"="mine"]->.m5;.m5 out count;
    way(around:5000,{lat},{lon})["man_made"="wastewater_plant"]->.w5;.w5 out count;
    way(around:5000,{lat},{lon})["landuse"="farmland"]->.f5;.f5 out count;
    way(around:5000,{lat},{lon})["highway"]->.h5;.h5 out count;

    node(around:10000,{lat},{lon})["man_made"="mine"]->.m10;.m10 out count;
    way(around:10000,{lat},{lon})["man_made"="wastewater_plant"]->.w10;.w10 out count;
    way(around:10000,{lat},{lon})["landuse"="farmland"]->.f10;.f10 out count;
    way(around:10000,{lat},{lon})["highway"]->.h10;.h10 out count;
    """
    
    keys = [
        'osm_mines_1000m', 'osm_wastewater_1000m', 'osm_farmland_1000m', 'osm_roads_1000m',
        'osm_mines_5000m', 'osm_wastewater_5000m', 'osm_farmland_5000m', 'osm_roads_5000m',
        'osm_mines_10000m', 'osm_wastewater_10000m', 'osm_farmland_10000m', 'osm_roads_10000m'
    ]
    
    headers = {
        'User-Agent': 'curl/8.7.1',
        'Accept': '*/*'
    }
    
    for attempt in range(retries):
        endpoint = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.get(endpoint, params={'data': query}, headers=headers, timeout=95)
            r.raise_for_status()
            elements = r.json().get('elements', [])
            
            if len(elements) == 12:
                results = {}
                for i, key in enumerate(keys):
                    count = int(elements[i].get('tags', {}).get('total', 0))
                    results[key] = count
                
                # Calculate totals
                results['osm_total_1000m'] = sum(results[k] for k in keys[0:4])
                results['osm_total_5000m'] = sum(results[k] for k in keys[4:8])
                results['osm_total_10000m'] = sum(results[k] for k in keys[8:12])
                
                results['osm_status'] = 'Success'
                
                # Verbose logging showing counts for all features
                print(f"[{station_num:3d}/{total_stations}] {station_id[:20]:20s} -> Success via {endpoint.split('/')[2]}")
                print(f"    Mines  (1k/5k/10k): [{results['osm_mines_1000m']}/{results['osm_mines_5000m']}/{results['osm_mines_10000m']}] | "
                      f"Waste: [{results['osm_wastewater_1000m']}/{results['osm_wastewater_5000m']}/{results['osm_wastewater_10000m']}]")
                print(f"    Farm   (1k/5k/10k): [{results['osm_farmland_1000m']}/{results['osm_farmland_5000m']}/{results['osm_farmland_10000m']}] | "
                      f"Roads: [{results['osm_roads_1000m']}/{results['osm_roads_5000m']}/{results['osm_roads_10000m']}]")
                return results
            else:
                raise ValueError(f"Expected 12 elements, got {len(elements)}")
                
        except Exception as e:
            sleep_time = 4 * (attempt + 1)
            if attempt < retries - 1:
                log.warning(f"[{station_num}/{total_stations}] Endpoint '{endpoint.split('/')[2]}' failed ({type(e).__name__}). Rotating endpoint in {sleep_time}s...")
                time.sleep(sleep_time)
            else:
                log.error(f"[{station_num}/{total_stations}] {station_id[:20]:20s} -> FAILED ALL ENDPOINTS & RETRIES (Error: {type(e).__name__})")
                
    # Fallback structure
    fallback = {k: 0 for k in keys}
    fallback['osm_total_1000m'] = 0
    fallback['osm_total_5000m'] = 0
    fallback['osm_total_10000m'] = 0
    fallback['osm_status'] = 'Failed'
    return fallback

---
## Perform Extraction

In [ ]:
total = len(unique_stations)
results = []
start_time = time.time()

log.info(f'Starting optimized OSM extraction for {total} stations...')

for idx, row in unique_stations.iterrows():
    lat, lon = row[LAT_COL], row[LON_COL]
    station = row[STATION_COL]
    
    # Single API call for all 12 features
    osm_data = fetch_osm_station(lat, lon, station, idx + 1, total)
    
    record = {STATION_COL: station, LAT_COL: lat, LON_COL: lon}
    record.update(osm_data)
    results.append(record)
    
    # Polite delay to respect public server limits
    time.sleep(2.0)

elapsed = time.time() - start_time
log.info(f'DONE in {elapsed/60:.1f} min')

In [ ]:
osm_df = pd.DataFrame(results)
display(osm_df.describe())
print(f"\nExtraction status:")
print(osm_df['osm_status'].value_counts())

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/osm.parquet'
osm_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(osm_df)} rows)')
print('\n=== DONE ===')
print('Output: osm.parquet')
print('Next: add this notebook output as dataset input for 01e (Merge)')